# Connect to RabbitMQ

In [1]:
from communication.typed_protocol_client import TypedRabbitMQClient
from communication.typed_protocol import LoadProgram, LoadTCPProgram, InjectWear, Play
from communication.rabbitmq import Rabbitmq
from pathlib import Path
import yaml
import numpy as np

def load_config(path: Path) -> dict:
    with path.open() as f:
        return yaml.safe_load(f)

connect_config = load_config(Path("../../../communication/connect.yml"))
connect_config["ip"] = "127.0.0.1"

typed_client = TypedRabbitMQClient(Rabbitmq(**connect_config))
typed_client.client.connect_to_server()

Waiting for RabbitMQ network routing... (5 attempts left)


# Move to some position

In [2]:
# Construct control message for loading a program
def mov_to_pos(position: list, vel: int = 60, acc: int = 80):
    msg = LoadProgram(joint_positions=position, max_velocity=vel, acceleration=acc)

    typed_client.publish(msg)
    # send control message for starting program
    typed_client.publish(Play())

position = [0, -np.pi/4, 0, -np.pi/4, 0, 0]

mov_to_pos(position)

# Begin listening of Wear Status

In [3]:
import time

class Timer:
    def __init__(self):
        self.start_time = 0
        self.final_time = None

    def begin_timing(self):
        self.start_time = time.time()
        self.final_time = None

    def stop_timing(self):
        if self.final_time is not None:
            return self.final_time
        self.final_time = time.time() - self.start_time
        return self.final_time

In [4]:
from communication.typed_protocol import WearStatus
def get_callback(timer: Timer):
    def callback(ws: WearStatus):
        if (ws.wear_detected):
            print(f"Time passed before wear detected: {timer.stop_timing()}")
    return callback

timer = Timer()

typed_client.subscribe(WearStatus, get_callback(timer), "temptemptemp")

import threading

threading.Thread(target=typed_client.client.start_consuming, daemon=True).start()

# Begin timer and inject wear

In [ ]:
def inject_wear(joints: list):
    msg = InjectWear(duration=180, fault_value=1, joints=joints)
    typed_client.publish(msg)

# Start timing
timer.begin_timing()

inject_wear([0, 1, 2, 3, 4, 5])

In [6]:
timer.final_time